In [44]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split 

Data Preprocessing:



Load the dataset into a suitable data structure (e.g., pandas DataFrame).

Handle missing values, if any.

Explore the dataset to understand its structure and attributes.

In [45]:
# Load the dataset
df = pd.read_csv('anime.csv')
df.head

<bound method NDFrame.head of        anime_id                                               name  \
0         32281                                     Kimi no Na wa.   
1          5114                   Fullmetal Alchemist: Brotherhood   
2         28977                                           Gintama°   
3          9253                                        Steins;Gate   
4          9969                                      Gintama&#039;   
...         ...                                                ...   
12289      9316       Toushindai My Lover: Minami tai Mecha-Minami   
12290      5543                                        Under World   
12291      5621                     Violence Gekiga David no Hoshi   
12292      6133  Violence Gekiga Shin David no Hoshi: Inma Dens...   
12293     26081                   Yasuji no Pornorama: Yacchimae!!   

                                                   genre   type episodes  \
0                   Drama, Romance, School, Supernatu

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [47]:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

Feature Extraction:



Decide on the features that will be used for computing similarity (e.g., genres, user ratings).

Convert categorical features into numerical representations if necessary.

Normalize numerical features if required.



In [48]:
df['rating'] = df['rating'].fillna(df['rating'].mean())

In [49]:
df['genre'] = df['genre'].fillna('Unknown')

In [50]:
df['type'] = df['type'].fillna('Unknown')

In [51]:
if df['episodes'].dtype == 'object':
    df['episodes'] = df['episodes'].replace('Unknown', 0).astype(int) # Assuming 'Unknown' means 0 episodes or can be imputed
else:
    df['episodes'] = df['episodes'].fillna(0) 


In [52]:
df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [53]:
df['genre'] = df['genre'].apply(lambda x: x.split(', ') if isinstance(x, str) else [])

In [54]:
mlb = MultiLabelBinarizer()
genre_features = pd.DataFrame(mlb.fit_transform(df['genre']), columns=mlb.classes_, index=df.index)

In [55]:
scaler = MinMaxScaler()
df['rating_normalized'] = scaler.fit_transform(df[['rating']])

In [56]:
features = pd.concat([genre_features, df[['rating_normalized']]], axis=1)

In [57]:
features.head()

,Action,Adventure,Cars,Comedy,Dementia,Demons,Drama,Ecchi,Fantasy,Game,...,Space,Sports,Super Power,Supernatural,Thriller,Unknown,Vampire,Yaoi,Yuri,rating_normalized
0,0,0,0,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,0,0.924370
1,1,1,0,0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0.911164
2,1,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.909964
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0.900360
4,1,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.899160


In [58]:
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 45 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Action             12294 non-null  int32  
 1   Adventure          12294 non-null  int32  
 2   Cars               12294 non-null  int32  
 3   Comedy             12294 non-null  int32  
 4   Dementia           12294 non-null  int32  
 5   Demons             12294 non-null  int32  
 6   Drama              12294 non-null  int32  
 7   Ecchi              12294 non-null  int32  
 8   Fantasy            12294 non-null  int32  
 9   Game               12294 non-null  int32  
 10  Harem              12294 non-null  int32  
 11  Hentai             12294 non-null  int32  
 12  Historical         12294 non-null  int32  
 13  Horror             12294 non-null  int32  
 14  Josei              12294 non-null  int32  
 15  Kids               12294 non-null  int32  
 16  Magic              122

Recommendation System:



Design a function to recommend anime based on cosine similarity.

Given a target anime, recommend a list of similar anime based on cosine similarity scores.

Experiment with different threshold values for similarity scores to adjust the recommendation list size.

In [59]:
cosine_sim = cosine_similarity(features)

In [60]:
cosine_sim_df = pd.DataFrame(cosine_sim, index=df['anime_id'], columns=df['anime_id'])

In [61]:
# --- 3. Recommendation System (Cosine Similarity Calculation and Function Definition) ---
print("\n--- Step 3: Recommendation System Setup ---")
print("Calculating cosine similarity matrix. This might take a moment...")
cosine_sim = cosine_similarity(features)

# Convert the numpy array to a Pandas DataFrame for easier lookup using anime_id
cosine_sim_df = pd.DataFrame(cosine_sim, index=df['anime_id'], columns=df['anime_id'])
print("Cosine similarity matrix calculated and stored in 'cosine_sim_df'.")
print(f"Shape of cosine_sim_df: {cosine_sim_df.shape}")


def get_recommendations(anime_id, cosine_sim_df, df, top_n=10, threshold=0.5):
    """
    Recommends similar anime based on cosine similarity.

    Args:
        anime_id (int): The ID of the target anime.
        cosine_sim_df (pd.DataFrame): The pre-calculated cosine similarity matrix.
        df (pd.DataFrame): The original DataFrame containing anime information.
        top_n (int): The number of top recommendations to return.
        threshold (float): The similarity score threshold for recommendations.

    Returns:
        pd.DataFrame: A DataFrame of recommended anime.
    """
    # Initialize recommended_anime_details to an empty DataFrame at the start
    recommended_anime_details = pd.DataFrame()

   
    print(f"\nInside get_recommendations function. Target anime_id received: {anime_id}")
    

    if anime_id not in cosine_sim_df.index:
        print(f"Error: Anime ID {anime_id} not found in the similarity matrix. Cannot provide recommendations.")
        return recommended_anime_details # Return the empty DataFrame immediately

    # Get the similarity scores for the given anime_id from the pre-calculated matrix
    sim_scores = cosine_sim_df[anime_id]

    # Sort similarity scores in descending order
    sim_scores = sim_scores.sort_values(ascending=False)

    # Remove the target anime itself from the recommendations list
    sim_scores = sim_scores.drop(anime_id, errors='ignore') # 'errors='ignore'' prevents error if anime_id not found

    # Apply the similarity threshold
    sim_scores = sim_scores[sim_scores >= threshold]

    
    print(f" Number of recommendations after thresholding: {len(sim_scores)}")
    
    if not sim_scores.empty:
        top_recommended_anime_ids = sim_scores.head(top_n).index

        # Retrieve full details for the recommended anime from the original DataFrame
        recommended_anime_details = df[df['anime_id'].isin(top_recommended_anime_ids)].set_index('anime_id')

        # Add the similarity score to the recommended anime details
        # Ensure scores are aligned to the fetched anime details
        recommended_anime_details['similarity_score'] = sim_scores.loc[recommended_anime_details.index]
    else:
        # If sim_scores is empty, recommended_anime_details remains an empty DataFrame
        print(f"No recommendations found for anime_id {anime_id} with threshold {threshold}.")

    # This single return statement at the end ensures recommended_anime_details is always bound
    return recommended_anime_details.sort_values(by='similarity_score', ascending=False)

# --- 4. Example Usage and Demonstration ---
print("\n--- Step 4: Demonstrating Recommendations ---")

# Define the anime_id for which you want recommendations
# This variable is crucial for the function call!
example_anime_id = 32281 # This is 'Kimi no Na wa.' (Your Name.)

print(f"\nAttempting to get recommendations for Anime ID: {example_anime_id} (Kimi no Na wa.)")

# Call the recommendation function with the example_anime_id
print("\n--- Recommendations with Threshold = 0.7 (Top 5) ---")
recommendations_high_threshold = get_recommendations(example_anime_id, cosine_sim_df, df, top_n=5, threshold=0.7)
if not recommendations_high_threshold.empty:
    print(recommendations_high_threshold[['name', 'genre', 'rating', 'similarity_score']])
else:
    print("No recommendations found with the given high threshold and top_n.")

print("\n--- Recommendations with Threshold = 0.5 (Top 10) ---")
recommendations_low_threshold = get_recommendations(example_anime_id, cosine_sim_df, df, top_n=10, threshold=0.5)
if not recommendations_low_threshold.empty:
    print(recommendations_low_threshold[['name', 'genre', 'rating', 'similarity_score']])
else:
    print("No recommendations found with the given low threshold and top_n.")



--- Step 3: Recommendation System Setup ---
Calculating cosine similarity matrix. This might take a moment...
Cosine similarity matrix calculated and stored in 'cosine_sim_df'.
Shape of cosine_sim_df: (12294, 12294)

--- Step 4: Demonstrating Recommendations ---

Attempting to get recommendations for Anime ID: 32281 (Kimi no Na wa.)

--- Recommendations with Threshold = 0.7 (Top 5) ---

Inside get_recommendations function. Target anime_id received: 32281
 Number of recommendations after thresholding: 156
                                                       name  \
anime_id                                                      
547                             Wind: A Breath of Heart OVA   
546                            Wind: A Breath of Heart (TV)   
14669                 Aura: Maryuuin Kouga Saigo no Tatakai   
28725                         Kokoro ga Sakebitagatterunda.   
6351      Clannad: After Story - Mou Hitotsu no Sekai, K...   

                                               

Evaluation:



Split the dataset into training and testing sets.

Evaluate the recommendation system using appropriate metrics such as precision, recall, and F1-score.

Analyze the performance of the recommendation system and identify areas of improvement.



In [62]:
anime_ids = df['anime_id'].tolist()
train_anime_ids, test_anime_ids = train_test_split(anime_ids, test_size=0.2, random_state=42)

print(f"Total anime IDs: {len(anime_ids)}")
print(f"Test anime IDs (for evaluation): {len(test_anime_ids)}")

Total anime IDs: 12294
Test anime IDs (for evaluation): 2459


In [63]:
def get_ground_truth_relevant_items(target_anime_id, cosine_sim_df, df, ground_truth_threshold=0.95):
    if target_anime_id not in cosine_sim_df.index:
        return set() # Return an empty set if target not found

    sim_scores = cosine_sim_df[target_anime_id]
    # Exclude itself
    sim_scores = sim_scores.drop(target_anime_id, errors='ignore')
    # Filter by very high similarity for ground truth
    relevant_items = sim_scores[sim_scores >= ground_truth_threshold]
    return set(relevant_items.index.tolist())


In [64]:
RECOMMENDATION_TOP_N = 10
RECOMMENDATION_THRESHOLD = 0.5
GROUND_TRUTH_THRESHOLD = 0.95

In [65]:
all_precisions = []
all_recalls = []
all_f1_scores = []

In [66]:
num_anime_to_evaluate = min(100, len(test_anime_ids)) # Evaluate max 100 anime for demonstration
print(f"Evaluating metrics for {num_anime_to_evaluate} test anime...")

for i, test_anime_id in enumerate(test_anime_ids[:num_anime_to_evaluate]):
    # Get ground truth relevant items
    ground_truth_relevant_items = get_ground_truth_relevant_items(test_anime_id, cosine_sim_df, df, GROUND_TRUTH_THRESHOLD)

    # Get recommendations from our system
    recommended_df = get_recommendations(test_anime_id, cosine_sim_df, df, 
                                          top_n=RECOMMENDATION_TOP_N, 
                                          threshold=RECOMMENDATION_THRESHOLD)
    recommended_items = set(recommended_df.index.tolist())


Evaluating metrics for 100 test anime...

Inside get_recommendations function. Target anime_id received: 17209
 Number of recommendations after thresholding: 1497

Inside get_recommendations function. Target anime_id received: 173
 Number of recommendations after thresholding: 1148

Inside get_recommendations function. Target anime_id received: 3616
 Number of recommendations after thresholding: 1555

Inside get_recommendations function. Target anime_id received: 18799
 Number of recommendations after thresholding: 668

Inside get_recommendations function. Target anime_id received: 18831
 Number of recommendations after thresholding: 510

Inside get_recommendations function. Target anime_id received: 23319
 Number of recommendations after thresholding: 1922

Inside get_recommendations function. Target anime_id received: 2983
 Number of recommendations after thresholding: 310

Inside get_recommendations function. Target anime_id received: 21797
 Number of recommendations after threshold

In [67]:
# Precision
precision = np.nan_to_num(np.divide(true_positives, RECOMMENDATION_TOP_N))

In [68]:
# Recall
recall_denominator = len(ground_truth_relevant_items)
recall = np.nan_to_num(np.divide(true_positives, recall_denominator))
    

In [69]:
# F1-Score
f1_denominator = precision + recall
f1 = np.nan_to_num(np.divide(2 * (precision * recall), f1_denominator))
    

In [70]:
all_precisions.append(precision)
all_recalls.append(recall)
all_f1_scores.append(f1)

In [71]:
avg_precision = np.mean(all_precisions)
avg_recall = np.mean(all_recalls)
avg_f1_score = np.mean(all_f1_scores)

In [72]:
print(f"\n--- Evaluation Results (Simulated & without explicit if-else) ---")
print(f"Recommendation Top N: {RECOMMENDATION_TOP_N}")
print(f"Recommendation Threshold: {RECOMMENDATION_THRESHOLD}")
print(f"Ground Truth Similarity Threshold: {GROUND_TRUTH_THRESHOLD}")
print(f"Average Precision: {avg_precision:.4f}")
print(f"Average Recall: {avg_recall:.4f}")
print(f"Average F1-Score: {avg_f1_score:.4f}")


--- Evaluation Results (Simulated & without explicit if-else) ---
Recommendation Top N: 10
Recommendation Threshold: 0.5
Ground Truth Similarity Threshold: 0.95
Average Precision: 1.0000
Average Recall: 0.0562
Average F1-Score: 0.1064


In [73]:
print("\n--- Analysis of Performance and Areas for Improvement ---")
print("1. Performance Analysis:")
print(f"   - The calculated metrics (Precision: {avg_precision:.4f}, Recall: {avg_recall:.4f}, F1-Score: {avg_f1_score:.4f}) reflect how well our content-based system identifies and retrieves other anime that are highly similar in their genre and rating features.")
print(f"   - These scores are highly influenced by the chosen `RECOMMENDATION_TOP_N` (how many items we recommend), `RECOMMENDATION_THRESHOLD` (minimum similarity for a recommendation), and especially the `GROUND_TRUTH_THRESHOLD` (how we define 'relevant' items for the ground truth).")
print(f"   - A higher `GROUND_TRUTH_THRESHOLD` (like 0.95 used here) makes the 'relevant' set very strict, potentially leading to lower recall if the system struggles to find many *extremely* similar items within the top N recommendations.")
print("   - It's crucial to remember that this is a simulated evaluation of content similarity, not a direct measure of user satisfaction, as actual user interaction data is not available.")



--- Analysis of Performance and Areas for Improvement ---
1. Performance Analysis:
   - The calculated metrics (Precision: 1.0000, Recall: 0.0562, F1-Score: 0.1064) reflect how well our content-based system identifies and retrieves other anime that are highly similar in their genre and rating features.
   - These scores are highly influenced by the chosen `RECOMMENDATION_TOP_N` (how many items we recommend), `RECOMMENDATION_THRESHOLD` (minimum similarity for a recommendation), and especially the `GROUND_TRUTH_THRESHOLD` (how we define 'relevant' items for the ground truth).
   - A higher `GROUND_TRUTH_THRESHOLD` (like 0.95 used here) makes the 'relevant' set very strict, potentially leading to lower recall if the system struggles to find many *extremely* similar items within the top N recommendations.
   - It's crucial to remember that this is a simulated evaluation of content similarity, not a direct measure of user satisfaction, as actual user interaction data is not available.


In [74]:
print("\n2. Areas for Improvement:")
print("   a. **Feature Engineering:** Extend the feature set beyond genre and rating. Consider 'type' (Movie, TV, OVA), 'episodes' count, popularity metrics ('members'), or even external data like studio, themes, or voice actors (if available and quantifiable).")
print("   b. **Feature Weighting/Learning:** Explore methods to assign different weights to different features (e.g., genre might be more important than rating for some users). This could involve learning optimal weights or using domain expertise.")
print("   c. **Recommendation Diversity:** Implement strategies to increase the diversity of recommendations. Pure similarity can lead to very homogeneous lists. Techniques like Maximal Marginal Relevance (MMR) can balance relevance with diversity.")
print("   d. **Real-world User Data:** The most significant improvement would be to acquire explicit user-anime interaction data (e.g., individual user ratings) to enable a true collaborative filtering approach or a hybrid model.")
print("      - With user data, matrix factorization models (e.g., SVD, NMF) or deep learning models could be employed for more accurate and personalized recommendations.")
print("   e. **Scalability:** For very large datasets, pre-calculating the full similarity matrix and iterating through all items can be inefficient.")
print("      - Explore Approximate Nearest Neighbor (ANN) search algorithms (e.g., Faiss, Annoy, NMSLIB) to efficiently find top-N similar items without exhaustive search, improving performance for massive catalogs.")
print("   f. **Temporal Dynamics:** If interaction data were available, considering the recency of interactions or trends in user preferences could improve recommendations over time.")


2. Areas for Improvement:
   a. **Feature Engineering:** Extend the feature set beyond genre and rating. Consider 'type' (Movie, TV, OVA), 'episodes' count, popularity metrics ('members'), or even external data like studio, themes, or voice actors (if available and quantifiable).
   b. **Feature Weighting/Learning:** Explore methods to assign different weights to different features (e.g., genre might be more important than rating for some users). This could involve learning optimal weights or using domain expertise.
   c. **Recommendation Diversity:** Implement strategies to increase the diversity of recommendations. Pure similarity can lead to very homogeneous lists. Techniques like Maximal Marginal Relevance (MMR) can balance relevance with diversity.
   d. **Real-world User Data:** The most significant improvement would be to acquire explicit user-anime interaction data (e.g., individual user ratings) to enable a true collaborative filtering approach or a hybrid model.
      - With

Interview Questions:

1. Can you explain the difference between user-based and item-based collaborative filtering?
 
-->User-Based Collaborative Filtering (User-User CF):

Concept: This approach recommends items to a target user by identifying other users who have similar tastes or preferences. It operates on the principle that "users who agreed in the past will agree again in the future."


Item-Based Collaborative Filtering (Item-Item CF):

Concept: This approach recommends items to a target user based on the similarity between items themselves. It works on the idea that "if a user liked item A, they are likely to like other items similar to item A." Item similarity is determined by how users collectively rate or interact with items.


2. What is collaborative filtering, and how does it work?

-->Collaborative Filtering (CF) is a fundamental technique in recommendation systems that makes predictions about a user's interests by leveraging the collective preferences or taste information from many users. Its core principle is that if multiple people show similar tastes or behaviors (collaborate), their preferences can be used to make recommendations to each other.

How it Works (General Principle):

Collaborative filtering typically operates through the following steps:

Data Collection
Similarity Calculation
Prediction and Recommendation Generation